# Module 7 — Deep Learning for Campus Movement Prediction

This module extends the campus movement project from Module 6 into a deep learning setting. Earlier modules focused on probabilistic models such as maximum likelihood estimation, conditional probability learning, and EM for hidden activity modes. In this module, the goal is to build, train, and analyze neural network models for predicting campus movement.

The script is designed to satisfy all major deep learning requirements in one integrated system. It includes:
- a feedforward neural network for static prediction,
- optimization experiments with multiple learning rates,
- generalization analysis through overfitting and regularization,
- an advanced recurrent neural network architecture for sequential prediction,
- and an application design component that shows how the trained model could be used in practice.

A key feature of this module is the integration of **athlete sports schedules**, specifically for **Track and Field** and **Baseball**. This makes the synthetic world more realistic and gives the models richer patterns to learn.

---

## Overall Goal of the Module

The deep learning system is built around two related prediction tasks.

### Static prediction task
Predict the main campus location from:
- student type,
- sport,
- day of week,
- time slot.

Formally, the feedforward model approximates:

\[
P(\text{main\_location} \mid \text{student\_type}, \text{sport}, \text{day\_of\_week}, \text{time\_slot})
\]

### Sequential prediction task
Predict the next main location from:
- previous locations earlier in the day,
- student type,
- sport,
- day of week.

This second task is modeled with an RNN because campus movement is naturally sequential. A student’s next location often depends on where they have already been.

---

## Reproducibility and Device Setup

The code begins by setting fixed random seeds for:
- Python’s `random`,
- NumPy,
- and PyTorch.

This ensures reproducibility in:
- dataset generation,
- train-validation-test splitting,
- and neural network initialization.

The script also detects whether a GPU is available and selects:
- `"cuda"` if available,
- otherwise `"cpu"`.

This is important because training neural networks can be computationally expensive, especially for repeated experiments such as optimization tuning and recurrent modeling.

---

## Canonical Project Schema

The schema extends the corrected campus world from Module 6.

### Student types
- Regular Student
- Copa
- Athlete

### Time slots
These divide the day into structured movement intervals, such as:
- 07:00–08:00
- 08:00–09:30
- …
- 18:00–21:00

### Days of the week
The model includes weekdays:
- Monday
- Tuesday
- Wednesday
- Thursday
- Friday

### Sports
Athletes are further subdivided into:
- None
- TrackAndField
- Baseball

This is a major addition because not all athletes move through campus in the same way. Their sport affects where they are likely to be.

### Location hierarchy
The location design keeps the semantic correction from Module 6:
- **VillagePark** is a general social campus area.
- **Track** is a dedicated athletic training location.

This distinction is crucial because it prevents the model from incorrectly learning that all athletic afternoon behavior happens in a general social area.

---

## Sports Schedules Integrated Into the World

One of the strongest features of this module is that it does not treat athletes as one homogeneous group. Instead, it explicitly encodes realistic sports schedules for Track & Field and Baseball.

### Track and Field schedule
The schedule includes:
- Tuesday and Thursday morning gym sessions,
- afternoon track practice on all weekdays.

Examples:
- Tuesday 07:00–08:00 → StudentCenter / Gym
- Monday–Friday 15:00–15:30 → Track
- Monday–Friday 16:00–17:30 → Track

### Baseball schedule
The schedule includes:
- off-campus games,
- gym blocks,
- batting practice in StudentCenter.

Examples:
- Monday 13:00–14:30 → OffCampus
- Monday 16:00–17:30 → StudentCenter / Gym
- Tuesday and Thursday 13:00–14:30 → StudentCenter / AthleteTrainers
- Friday 07:00–14:30 → StudentCenter / Gym

### Why schedule integration matters
These schedules are not just decorative. They are directly used to modify the synthetic world. The function `apply_sport_schedule_boosts()` increases the probability of scheduled locations for athlete samples. This means:
- the data generation process is more realistic,
- the model learns meaningful sport-specific patterns,
- and prediction becomes more faithful to real behavior.

Without this schedule integration, the model would only learn generic athlete behavior. With it, the model can distinguish between Track & Field and Baseball movement patterns.

---

## Base World Probabilities

The script defines the general campus movement distributions for each student type and time slot. These distributions are inherited from the probabilistic world used in previous modules, with the updated distinction between Track and VillagePark.

### Main location weights
These specify:

\[
P(\text{main\_location} \mid \text{student\_type}, \text{time\_slot})
\]

For example:
- athletes in late afternoon are strongly associated with Track,
- Copa students are more associated with PlayHouse and BoulevardStudios,
- regular students are more associated with academic spaces like ThayerHall, WestPenn, and Library.

### Inner location weights
These specify:

\[
P(\text{inner\_location} \mid \text{main\_location}, \text{student\_type})
\]

These weights refine building-level behavior into room-level behavior. For example:
- StudentCenter for athletes favors Gym and AthleteTrainers,
- PlayHouse for Copa favors ActingStudios and FilmStudios,
- ThayerHall for regular students favors RegularClassroom and ComputerLab.

These probabilities are used during synthetic data generation to create realistic labeled datasets.

---

## Synthetic Data Generation Helpers

The code defines helper functions for turning the probability tables and sports schedules into actual observations.

### `weighted_choice`
This function performs weighted random sampling from a dictionary of categories and weights.

### `sample_sport`
This function assigns a sport to an athlete. Non-athletes always receive `"None"`.

### `apply_sport_schedule_boosts`
This function is central to the realism of the dataset. It takes the base location weights and modifies them when a sport schedule applies. For example:
- if a Track & Field athlete is scheduled to be at Track at 16:00–17:30, the Track weight is increased strongly,
- if a Baseball athlete is scheduled for batting practice in StudentCenter, StudentCenter receives an additional boost.

### `sample_main_location`
This samples a main location after the schedule-based adjustments are applied.

### `sample_inner_location`
This samples an inner location and also respects sport schedules. For example:
- Track & Field athletes at scheduled track times are very likely to end up at `Track > Track`,
- Baseball athletes at gym or batting-practice times are strongly pushed toward the scheduled inner location.

These helper functions create a synthetic world that is much richer than a simple random generator. The result is structured data with strong temporal and contextual patterns that neural networks can learn from.

---

## Dataset Generation

Two datasets are generated.

### Static dataset
The function `generate_static_dataset()` creates independent rows where each row contains:
- student type,
- sport,
- day of week,
- time slot,
- main location,
- inner location.

This dataset is used for the feedforward neural network. In this setting, each sample is treated independently.

### Sequential dataset
The function `generate_daily_sequences()` creates time-ordered daily movement sequences for synthetic students. Each record contains:
- student ID,
- student type,
- sport,
- day of week,
- time slot,
- previous main location,
- current main location,
- inner location.

This dataset is used for the recurrent model. Here, the main idea is that movement is sequential: previous locations influence the next one.

The script generates:
- 40,000 static observations,
- and daily sequences for 2,500 students across all weekdays.

This gives both models enough data to learn patterns reliably.

---

## Part 1 — Feedforward Model (21.1–21.2)

The first neural model is a feedforward network for static prediction.

### Prediction task
The model predicts the main campus location from:
- student type,
- sport,
- day of week,
- time slot.

### Data preparation
The input features are categorical, so the script uses `OneHotEncoder` to transform them into a numeric vector. The target labels (main locations) are encoded with `LabelEncoder`.

The data is then split into:
- training set,
- validation set,
- test set.

This makes it possible to:
- train the model,
- tune decisions using validation performance,
- and report final generalization on the test set.

### Computation graph
The script explicitly describes the feedforward computation graph as:

\[
\text{Input} \rightarrow \text{Linear} \rightarrow \text{ReLU} \rightarrow \text{Linear} \rightarrow \text{ReLU} \rightarrow \text{Output logits} \rightarrow \text{Softmax probabilities}
\]

This is a standard multilayer perceptron for tabular classification.

### Dataset and DataLoader helpers
The class `TabularDataset` converts NumPy arrays into PyTorch tensors. DataLoaders are then used to provide mini-batches during training.

### Feedforward architecture
The model `FeedForwardNet` contains:
- an input layer,
- two hidden layers,
- ReLU activations,
- optional dropout,
- and a final linear output layer.

The default hidden sizes are:
- 64
- 32

The final layer outputs logits over all main locations.

### Training function
The function `train_feedforward_model()` performs training using:
- cross-entropy loss,
- Adam optimizer,
- mini-batch gradient descent,
- validation monitoring,
- and early stopping.

At each epoch, it records:
- training loss,
- validation loss,
- training accuracy,
- validation accuracy,
- training macro-F1,
- validation macro-F1.

This gives a full picture of learning dynamics.

### Baseline feedforward training
The script trains a baseline feedforward model with:
- hidden dimensions `(64, 32)`,
- learning rate `0.01`,
- no dropout,
- no weight decay.

It then evaluates the model on the test set and saves:
- `module7_ff_loss_curves.png`
- `module7_ff_accuracy_curves.png`

These plots show how training and validation metrics evolve over time.

---

## Part 2 — Optimization (21.4)

The next section studies how optimization behavior changes with the learning rate.

### Learning rates tested
The same feedforward architecture is trained with:
- `0.1`
- `0.01`
- `0.001`

### Purpose
This experiment shows how gradient-based optimization depends on step size.

- A **high learning rate** can overshoot good parameter regions and make loss unstable.
- A **low learning rate** can be very slow and may require many epochs to make progress.
- A **moderate learning rate** often balances convergence speed and stability.

### Implementation
For each learning rate:
- a new feedforward network is initialized,
- training is run for up to 25 epochs,
- early stopping is applied,
- validation loss, accuracy, and macro-F1 are recorded.

The results are stored in `lr_results` and summarized numerically.

A plot is then saved as:

```text
module7_learning_rate_comparison.png

## Optimization Visualization

This graph shows validation loss curves across different learning rates, making optimization behavior easy to compare visually.

### What this section demonstrates

This part satisfies the optimization requirement by showing that deep learning performance is not determined only by architecture. Optimization hyperparameters, especially learning rate, have a major effect on whether training converges well or fails.

---

## Part 3 — Generalization (21.5)

The next section studies generalization, specifically:

- how overfitting occurs,
- and how regularization can reduce it.

### Overfitting setup

To create an overfitting scenario, the script intentionally:

- reduces the training data to only 3,000 samples,
- and uses a much larger feedforward network with hidden layers (256, 128).

This larger model has much more capacity relative to the amount of data.

### Overfitting model

The overfitting model is trained without dropout or weight decay. As expected, it can fit the small training set very well, but validation performance becomes worse. This is the classic pattern of overfitting:

- training loss decreases strongly,
- validation loss does not improve at the same rate and may rise.

### Regularized model

The script then trains a second model on the same small dataset, but this time with regularization:

- smaller hidden dimensions (64, 32),
- dropout 0.30,
- weight decay 1e-4,
- early stopping.

This model is expected to generalize better because it is less likely to memorize idiosyncratic details of the small training subset.

### Saved plots

The section saves:

- `module7_overfitting_demo.png`
- `module7_regularization_demo.png`

These plots compare training and validation loss and make the generalization story very clear.

### Key lesson

This part demonstrates that:

- increasing model capacity can improve fit,
- but too much capacity without enough data leads to overfitting,
- and regularization methods such as dropout, weight decay, and early stopping are effective tools for improving generalization.

---

## Part 4 — Advanced Architecture: RNN (21.3 / 21.6)

The script then moves from static prediction to sequential prediction using an RNN.

### Why an RNN is appropriate

The feedforward model predicts location from static context only. However, campus movement is not purely static. A student’s next location often depends on where they have already been earlier in the day.

This makes sequential modeling a more natural fit, and the script implements this using a GRU-based recurrent neural network.

### Sequence construction

The sequential dataset is reorganized so that each training example contains:

- student type,
- sport,
- day of week,
- a prefix of previous main locations,
- and the next main location as the target.

Each sequence starts with a special token:

- `START`

This allows the model to make the first prediction of the day as well.

### Encoding

The script constructs:

- location-to-index mappings,
- type-to-index mappings,
- sport-to-index mappings,
- day-to-index mappings.

The sequence target is then the next location index.

### SequenceDataset and collate function

The custom sequence dataset returns:

- metadata indices,
- sequence tokens,
- target location.

Because sequences have variable lengths, the script defines `collate_sequence_batch()` to:

- pad sequences to the same length in a batch,
- keep track of original lengths.

This is standard practice for sequence modeling.

### RNN architecture

The class `CampusRNN` contains:

- an embedding layer for locations,
- embedding layers for student type, sport, and day,
- a GRU recurrent layer,
- a feedforward classifier on top.

The GRU processes the previous-location sequence and outputs a hidden state representing temporal context. This hidden state is concatenated with the metadata embeddings and passed through a small dense network to predict the next location.

### Why this architecture is effective

This design allows the model to combine:

- temporal information from the sequence,
- with static context from type, sport, and day.

That is a strong design choice because movement patterns depend on both.

### Training and evaluation

The functions `train_rnn_model()` and `evaluate_rnn()` mirror the feedforward training loop:

- cross-entropy loss,
- Adam optimizer,
- validation monitoring,
- early stopping.

The model is then evaluated on the RNN test set and the following plots are saved:

- `module7_rnn_loss_curves.png`
- `module7_rnn_accuracy_curves.png`

### What this section demonstrates

This part satisfies the advanced architecture requirement because it moves beyond a simple feedforward network to a recurrent sequential model. It also shows that sequence-aware models are more appropriate when the task involves temporal movement.

---

## Model Comparison

After both models are trained, the script creates a comparison table with:

- test loss,
- test accuracy,
- test macro-F1.

The comparison is between:

- Feedforward Baseline
- RNN Sequential Model

This is important because it directly evaluates whether adding sequential structure improves performance. In many movement tasks, the RNN should be more powerful because it uses history instead of relying only on static context.

---

## Part 5 — Application Design (21.7–21.8)

The module concludes with an application-oriented design component.

### User-facing prediction helper

The function `predict_static_location_distribution()` acts as a frontend helper. It takes:

- student type,
- sport,
- day of week,
- time slot,

and returns a ranked probability distribution over main campus locations.

### Example use case

A user can query:

- Athlete  
- TrackAndField  
- Tuesday  
- 16:00–17:30  

The model will then output a sorted list of likely main locations, with probabilities.

### Why this matters

This transforms the trained feedforward model into a usable application interface. Instead of only producing test metrics, the project demonstrates how the model could support:

- facility planning,
- staffing decisions,
- campus analytics,
- student movement forecasting.

This satisfies the application design requirement because the model is clearly connected to a practical end use.

---

## Final Performance Metrics

The script finishes by generating a classification report for the feedforward baseline model. This includes per-class precision, recall, and F1. Reporting these metrics is useful because overall accuracy alone can hide class-specific weaknesses.

For example, some locations may be easier to predict because they are tightly tied to schedules, while others may be more ambiguous. The classification report helps reveal those differences.

---

## Reflection Questions

### How did depth improve performance?

Depth improved performance by allowing the feedforward network to learn more complex interactions between input variables. A shallow model might only capture simple direct associations, such as:

> “Track athletes in late afternoon often go to Track.”

A deeper model can learn more layered combinations such as:

- athlete + baseball + Friday + midday → StudentCenter / Gym  
- athlete + track + Tuesday morning → StudentCenter / Gym  
- Copa + afternoon + specific day → PlayHouse or BoulevardStudios  

In other words, extra depth gives the network greater representational power. It helps the model build nonlinear transformations of the one-hot encoded inputs and capture more subtle dependencies.

However, depth only improves performance when it is matched to enough data and appropriate regularization. In the overfitting experiment, increasing depth and width too much on a small dataset led to memorization rather than real improvement.

---

### When did overfitting occur?

Overfitting occurred in the experiment where the model was intentionally made too large relative to the amount of training data. Specifically, the script used:

- a small training subset of only 3,000 samples,
- and a large network with hidden layers (256, 128).

In this setting, the model had enough capacity to fit the training set extremely well, but it did not generalize as effectively to validation and test data.

The signs of overfitting were:

- training loss kept decreasing,
- training accuracy became very strong,
- but validation loss and validation performance did not improve in the same way.

This is the classic deep learning overfitting pattern: the model memorizes training data instead of learning generalizable patterns.

---

### What optimization challenges arose?

Several optimization challenges appeared during training.

**1. Learning rate selection**

- A high learning rate (e.g., 0.1) can make optimization unstable.
- A very low learning rate (e.g., 0.001) can slow convergence significantly.

**2. Convergence vs. stability tradeoff**

- A moderate learning rate (e.g., 0.01) often provides the best balance.

**3. When to stop training**

- Early stopping was essential to prevent unnecessary training and reduce overfitting.

**Summary of challenges:**

- choosing an appropriate learning rate,
- controlling instability,
- deciding when to stop training.

---

### How did transfer learning change results?

In this script, traditional transfer learning was not explicitly implemented. The model does not use pretrained weights.

However, a form of **domain knowledge injection** exists through:

- structured sports schedules (Track & Field, Baseball).

This acts as prior knowledge, improving realism and helping the model learn stronger patterns.

If transfer learning had been used, it could have:

- improved convergence speed,
- enhanced performance on small datasets,
- improved generalization.

---

## Final Summary

Module 7 successfully extends the campus movement project into a deep learning framework.

It includes:

- a feedforward neural network for static location prediction,
- optimization experiments with multiple learning rates,
- explicit demonstrations of overfitting and regularization,
- a GRU-based recurrent neural network for sequential movement prediction,
- and a user-facing application design component.

The integration of sports schedules makes the synthetic world significantly more realistic and gives both the feedforward and sequential models rich patterns to learn.

The result is a complete deep learning system that is both technically solid and closely tied to the semester-long campus movement project.

In [1]:
import sys
print(sys.executable)

/opt/homebrew/opt/python@3.13/bin/python3.13


In [2]:
# ============================================================
# MODULE 7 — Deep Learning for Campus Movement Prediction
# Continuation of Module 6
#
# Goal:
# Build, train, and analyze a deep neural network system
# for the same campus movement project.
#
# This single Python script satisfies:
# Part 1: Feedforward Model (21.1–21.2)
# Part 2: Optimization (21.4)
# Part 3: Generalization (21.5)
# Part 4: Advanced Architecture (RNN) (21.3 / 21.6)
# Part 5: Application Design (21.7–21.8)
#
# Integrated athlete sports schedules:
# - Track and Field
# - Baseball
#
# Main static task:
# Predict main_location from:
#   student_type, sport, day_of_week, time_slot
#
# Sequential task:
# Predict next main_location from previous locations in the day
# using an RNN.
#
# Output files:
# - module7_ff_loss_curves.png
# - module7_ff_accuracy_curves.png
# - module7_learning_rate_comparison.png
# - module7_overfitting_demo.png
# - module7_regularization_demo.png
# - module7_rnn_loss_curves.png
# - module7_rnn_accuracy_curves.png
# ============================================================
import random
from collections import defaultdict

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Optional check
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# ============================================================
# 0. Reproducibility
# ============================================================

RANDOM_SEED = 7
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ============================================================
# 1. Canonical project schema
# ============================================================

STUDENT_TYPES = [
    "Regular Student",
    "Copa",
    "Athlete"
]

TIME_SLOTS = [
    "07:00-08:00",
    "08:00-09:30",
    "09:40-11:10",
    "11:20-12:50",
    "13:00-14:30",
    "15:00-15:30",
    "16:00-17:30",
    "18:00-21:00"
]

DAY_OF_WEEK = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday"
]

SPORTS = [
    "None",
    "TrackAndField",
    "Baseball"
]

# VillagePark is a central hangout area.
# Track is now an explicit athletic training location.
LOCATION_HIERARCHY = {
    "LawrenceHall": [
        "DanceStudios",
        "DiningHall",
        "CopaTrainers",
        "HarryPotterRoom"
    ],
    "ThayerHall": [
        "RegularClassroom",
        "ComputerLab"
    ],
    "WestPenn": [
        "AnimationStudio",
        "ScreenWritingStudio",
        "RegularClassroom"
    ],
    "StudentCenter": [
        "Gym",
        "AthleteTrainers",
        "GameRoom",
        "CareerCenter"
    ],
    "Library": [
        "StudyRooms"
    ],
    "VillagePark": [
        "VillagePark"
    ],
    "Track": [
        "Track"
    ],
    "PlayHouse": [
        "ActingStudios",
        "FilmStudios"
    ],
    "BoulevardStudios": [
        "BoulevardStudios"
    ],
    "BoulevardAppartments": [
        "MusicalTheaterStudios"
    ],
    "OffCampus": [
        "OffCampus"
    ]
}

MAIN_LOCATIONS = list(LOCATION_HIERARCHY.keys())

# ============================================================
# 2. Sports schedules integrated into the world
# ============================================================

# Track and Field
# Monday 3-5pm @track
# Tuesday 7-8am@gym then 3-5pm@track
# Wednesday 3-5pm@track
# Thursday 7-8am@gym then 3-5pm@track
# Friday 3-5pm@track

TRACK_SCHEDULE = {
    ("Tuesday",  "07:00-08:00"): ("StudentCenter", "Gym"),
    ("Thursday", "07:00-08:00"): ("StudentCenter", "Gym"),

    ("Monday",    "15:00-15:30"): ("Track", "Track"),
    ("Tuesday",   "15:00-15:30"): ("Track", "Track"),
    ("Wednesday", "15:00-15:30"): ("Track", "Track"),
    ("Thursday",  "15:00-15:30"): ("Track", "Track"),
    ("Friday",    "15:00-15:30"): ("Track", "Track"),

    ("Monday",    "16:00-17:30"): ("Track", "Track"),
    ("Tuesday",   "16:00-17:30"): ("Track", "Track"),
    ("Wednesday", "16:00-17:30"): ("Track", "Track"),
    ("Thursday",  "16:00-17:30"): ("Track", "Track"),
    ("Friday",    "16:00-17:30"): ("Track", "Track"),
}

# Baseball
# Monday: 1:05 PM -> Greentree Sportplex; 4:00 PM -> Gym
# Tuesday: 2:00 PM -> Gym; 1:00 PM -> Batting practice on SC 6th floor
# Wednesday: 1:05 PM -> Greentree Sportplex
# Thursday: 2:00 PM -> Gym; 1:00 PM -> Batting practice on SC 6th floor
# Friday: 7:00 AM – 3:00 PM -> Gym

BASEBALL_SCHEDULE = {
    ("Monday",    "13:00-14:30"): ("OffCampus", "OffCampus"),
    ("Wednesday", "13:00-14:30"): ("OffCampus", "OffCampus"),

    ("Monday",    "16:00-17:30"): ("StudentCenter", "Gym"),

    ("Tuesday",   "13:00-14:30"): ("StudentCenter", "AthleteTrainers"),
    ("Thursday",  "13:00-14:30"): ("StudentCenter", "AthleteTrainers"),

    ("Friday", "07:00-08:00"): ("StudentCenter", "Gym"),
    ("Friday", "08:00-09:30"): ("StudentCenter", "Gym"),
    ("Friday", "09:40-11:10"): ("StudentCenter", "Gym"),
    ("Friday", "11:20-12:50"): ("StudentCenter", "Gym"),
    ("Friday", "13:00-14:30"): ("StudentCenter", "Gym"),
}

# ============================================================
# 3. Base world probabilities (extended from Module 6)
# ============================================================

MAIN_LOCATION_WEIGHTS = {
    "Athlete": {
        "07:00-08:00": {
            "OffCampus": 5,
            "StudentCenter": 3,
            "LawrenceHall": 1,
            "ThayerHall": 1
        },
        "08:00-09:30": {
            "StudentCenter": 3,
            "LawrenceHall": 2,
            "Library": 2,
            "ThayerHall": 1,
            "VillagePark": 1,
            "OffCampus": 1
        },
        "09:40-11:10": {
            "StudentCenter": 4,
            "Library": 3,
            "ThayerHall": 2,
            "WestPenn": 1,
            "VillagePark": 1,
            "OffCampus": 1
        },
        "11:20-12:50": {
            "StudentCenter": 4,
            "LawrenceHall": 3,
            "BoulevardAppartments": 2,
            "Library": 2,
            "VillagePark": 1
        },
        "13:00-14:30": {
            "Library": 4,
            "ThayerHall": 3,
            "StudentCenter": 2,
            "WestPenn": 2,
            "VillagePark": 1
        },
        "15:00-15:30": {
            "Track": 3,
            "StudentCenter": 3,
            "WestPenn": 2,
            "Library": 1,
            "VillagePark": 1
        },
        "16:00-17:30": {
            "Track": 5,
            "StudentCenter": 2,
            "WestPenn": 2,
            "OffCampus": 1,
            "VillagePark": 1
        },
        "18:00-21:00": {
            "StudentCenter": 4,
            "OffCampus": 3,
            "BoulevardAppartments": 2,
            "VillagePark": 1,
            "Track": 1
        }
    },

    "Copa": {
        "07:00-08:00": {"OffCampus": 10},
        "08:00-09:30": {
            "LawrenceHall": 3,
            "PlayHouse": 3,
            "BoulevardStudios": 3,
            "StudentCenter": 1,
            "OffCampus": 1
        },
        "09:40-11:10": {
            "BoulevardStudios": 4,
            "PlayHouse": 3,
            "LawrenceHall": 2,
            "StudentCenter": 2,
            "Library": 1
        },
        "11:20-12:50": {
            "LawrenceHall": 4,
            "StudentCenter": 3,
            "Library": 2,
            "PlayHouse": 2,
            "BoulevardStudios": 1
        },
        "13:00-14:30": {
            "BoulevardStudios": 4,
            "ThayerHall": 2,
            "PlayHouse": 3,
            "Library": 2,
            "StudentCenter": 1
        },
        "15:00-15:30": {
            "PlayHouse": 3,
            "StudentCenter": 2,
            "Library": 2,
            "BoulevardStudios": 3
        },
        "16:00-17:30": {
            "PlayHouse": 4,
            "StudentCenter": 3,
            "BoulevardStudios": 3,
            "Library": 1
        },
        "18:00-21:00": {
            "StudentCenter": 3,
            "OffCampus": 3,
            "BoulevardAppartments": 2,
            "PlayHouse": 2
        }
    },

    "Regular Student": {
        "07:00-08:00": {"OffCampus": 10},
        "08:00-09:30": {
            "ThayerHall": 4,
            "WestPenn": 3,
            "Library": 2,
            "StudentCenter": 1
        },
        "09:40-11:10": {
            "ThayerHall": 4,
            "WestPenn": 4,
            "Library": 2,
            "StudentCenter": 1
        },
        "11:20-12:50": {
            "LawrenceHall": 3,
            "StudentCenter": 3,
            "Library": 3,
            "ThayerHall": 1,
            "WestPenn": 1
        },
        "13:00-14:30": {
            "ThayerHall": 3,
            "WestPenn": 3,
            "Library": 3,
            "StudentCenter": 2
        },
        "15:00-15:30": {
            "StudentCenter": 3,
            "Library": 3,
            "ThayerHall": 2,
            "WestPenn": 2
        },
        "16:00-17:30": {
            "StudentCenter": 3,
            "Library": 3,
            "BoulevardAppartments": 2,
            "OffCampus": 2,
            "WestPenn": 1
        },
        "18:00-21:00": {
            "OffCampus": 4,
            "BoulevardAppartments": 3,
            "StudentCenter": 2,
            "Library": 1
        }
    }
}

INNER_LOCATION_WEIGHTS = {
    "Athlete": {
        "StudentCenter": {
            "Gym": 4,
            "AthleteTrainers": 3,
            "GameRoom": 1,
            "CareerCenter": 1
        },
        "LawrenceHall": {
            "DiningHall": 4,
            "HarryPotterRoom": 1,
            "DanceStudios": 1,
            "CopaTrainers": 1
        },
        "Library": {"StudyRooms": 1},
        "VillagePark": {"VillagePark": 1},
        "Track": {"Track": 1},
        "PlayHouse": {"ActingStudios": 1, "FilmStudios": 1},
        "BoulevardStudios": {"BoulevardStudios": 1},
        "BoulevardAppartments": {"MusicalTheaterStudios": 1},
        "ThayerHall": {"RegularClassroom": 2, "ComputerLab": 1},
        "WestPenn": {"RegularClassroom": 2, "AnimationStudio": 1, "ScreenWritingStudio": 1},
        "OffCampus": {"OffCampus": 1}
    },

    "Copa": {
        "LawrenceHall": {
            "DanceStudios": 4,
            "DiningHall": 2,
            "CopaTrainers": 3,
            "HarryPotterRoom": 1
        },
        "PlayHouse": {"ActingStudios": 4, "FilmStudios": 3},
        "BoulevardStudios": {"BoulevardStudios": 1},
        "StudentCenter": {
            "CareerCenter": 2,
            "GameRoom": 2,
            "Gym": 1,
            "AthleteTrainers": 1
        },
        "BoulevardAppartments": {"MusicalTheaterStudios": 1},
        "Library": {"StudyRooms": 1},
        "ThayerHall": {"RegularClassroom": 1, "ComputerLab": 1},
        "WestPenn": {"AnimationStudio": 2, "ScreenWritingStudio": 3, "RegularClassroom": 1},
        "VillagePark": {"VillagePark": 1},
        "Track": {"Track": 1},
        "OffCampus": {"OffCampus": 1}
    },

    "Regular Student": {
        "ThayerHall": {"RegularClassroom": 4, "ComputerLab": 3},
        "WestPenn": {"RegularClassroom": 4, "AnimationStudio": 1, "ScreenWritingStudio": 1},
        "StudentCenter": {
            "GameRoom": 3,
            "CareerCenter": 2,
            "Gym": 1,
            "AthleteTrainers": 1
        },
        "Library": {"StudyRooms": 1},
        "LawrenceHall": {
            "DiningHall": 3,
            "HarryPotterRoom": 1,
            "DanceStudios": 1,
            "CopaTrainers": 1
        },
        "BoulevardAppartments": {"MusicalTheaterStudios": 1},
        "VillagePark": {"VillagePark": 1},
        "Track": {"Track": 1},
        "PlayHouse": {"ActingStudios": 1, "FilmStudios": 1},
        "BoulevardStudios": {"BoulevardStudios": 1},
        "OffCampus": {"OffCampus": 1}
    }
}

# ============================================================
# 4. Synthetic data generation helpers
# ============================================================

def weighted_choice(weight_dict: dict[str, float], rng: random.Random) -> str:
    items = [(k, v) for k, v in weight_dict.items() if v > 0]
    total = sum(v for _, v in items)
    r = rng.uniform(0, total)
    upto = 0.0
    for key, w in items:
        upto += w
        if upto >= r:
            return key
    return items[-1][0]


def sample_sport(student_type: str, rng: random.Random) -> str:
    if student_type != "Athlete":
        return "None"
    return rng.choice(["TrackAndField", "Baseball"])


def apply_sport_schedule_boosts(
    student_type: str,
    sport: str,
    day_of_week: str,
    time_slot: str,
    base_weights: dict[str, float]
) -> dict[str, float]:
    """
    Adjust location weights using explicit sports schedules.
    """
    weights = dict(base_weights)

    if student_type != "Athlete" or sport == "None":
        return weights

    if sport == "TrackAndField":
        if (day_of_week, time_slot) in TRACK_SCHEDULE:
            target_main, _ = TRACK_SCHEDULE[(day_of_week, time_slot)]
            weights[target_main] = weights.get(target_main, 0) + 12

    elif sport == "Baseball":
        if (day_of_week, time_slot) in BASEBALL_SCHEDULE:
            target_main, _ = BASEBALL_SCHEDULE[(day_of_week, time_slot)]
            weights[target_main] = weights.get(target_main, 0) + 12

            # Tue/Thu 13:00-14:30 includes batting practice + gym influence
            if day_of_week in ["Tuesday", "Thursday"] and time_slot == "13:00-14:30":
                weights["StudentCenter"] = weights.get("StudentCenter", 0) + 6

    return weights


def sample_main_location(
    student_type: str,
    sport: str,
    day_of_week: str,
    time_slot: str,
    rng: random.Random
) -> str:
    base_weights = MAIN_LOCATION_WEIGHTS[student_type][time_slot]
    adjusted_weights = apply_sport_schedule_boosts(
        student_type, sport, day_of_week, time_slot, base_weights
    )
    return weighted_choice(adjusted_weights, rng)


def sample_inner_location(
    student_type: str,
    sport: str,
    day_of_week: str,
    time_slot: str,
    main_location: str,
    rng: random.Random
) -> str:
    if student_type == "Athlete":
        if sport == "TrackAndField" and (day_of_week, time_slot) in TRACK_SCHEDULE:
            scheduled_main, scheduled_inner = TRACK_SCHEDULE[(day_of_week, time_slot)]
            if main_location == scheduled_main and rng.random() < 0.9:
                return scheduled_inner

        if sport == "Baseball" and (day_of_week, time_slot) in BASEBALL_SCHEDULE:
            scheduled_main, scheduled_inner = BASEBALL_SCHEDULE[(day_of_week, time_slot)]
            if main_location == scheduled_main and rng.random() < 0.8:
                return scheduled_inner

    if student_type in INNER_LOCATION_WEIGHTS and main_location in INNER_LOCATION_WEIGHTS[student_type]:
        weights = INNER_LOCATION_WEIGHTS[student_type][main_location]
        return weighted_choice(weights, rng)

    return rng.choice(LOCATION_HIERARCHY[main_location])


def generate_static_dataset(n_samples: int = 40000, seed: int = 7) -> pd.DataFrame:
    rng = random.Random(seed)
    rows = []

    for _ in range(n_samples):
        student_type = rng.choice(STUDENT_TYPES)
        day_of_week = rng.choice(DAY_OF_WEEK)
        time_slot = rng.choice(TIME_SLOTS)
        sport = sample_sport(student_type, rng)

        main_location = sample_main_location(
            student_type, sport, day_of_week, time_slot, rng
        )
        inner_location = sample_inner_location(
            student_type, sport, day_of_week, time_slot, main_location, rng
        )

        rows.append({
            "student_type": student_type,
            "sport": sport,
            "day_of_week": day_of_week,
            "time_slot": time_slot,
            "main_location": main_location,
            "inner_location": inner_location
        })

    return pd.DataFrame(rows)


def generate_daily_sequences(n_students: int = 2500, seed: int = 7) -> pd.DataFrame:
    rng = random.Random(seed)
    rows = []

    for sid in range(n_students):
        student_type = rng.choice(STUDENT_TYPES)
        sport = sample_sport(student_type, rng)

        for day_of_week in DAY_OF_WEEK:
            previous_main = None

            for time_slot in TIME_SLOTS:
                main_location = sample_main_location(
                    student_type, sport, day_of_week, time_slot, rng
                )
                inner_location = sample_inner_location(
                    student_type, sport, day_of_week, time_slot, main_location, rng
                )

                rows.append({
                    "student_id": sid,
                    "student_type": student_type,
                    "sport": sport,
                    "day_of_week": day_of_week,
                    "time_slot": time_slot,
                    "prev_main_location": previous_main if previous_main is not None else "START",
                    "main_location": main_location,
                    "inner_location": inner_location
                })

                previous_main = main_location

    return pd.DataFrame(rows)

# ============================================================
# 5. Generate datasets
# ============================================================

print("\n" + "=" * 70)
print("DATA GENERATION")
print("=" * 70)

df_static = generate_static_dataset(n_samples=40000, seed=RANDOM_SEED)
df_seq = generate_daily_sequences(n_students=2500, seed=RANDOM_SEED)

print("Static dataset shape:", df_static.shape)
print(df_static.head())

print("\nSequential dataset shape:", df_seq.shape)
print(df_seq.head(10))

print("\nAthlete sport distribution:")
print(df_static[df_static["student_type"] == "Athlete"]["sport"].value_counts(normalize=True))

# ============================================================
# 6. Feedforward model data preparation
# ============================================================

print("\n" + "=" * 70)
print("PART 1 — FEEDFORWARD MODEL (21.1–21.2)")
print("=" * 70)

X_static = df_static[["student_type", "sport", "day_of_week", "time_slot"]].copy()
y_static = df_static["main_location"].copy()

# Compatibility for different sklearn versions
try:
    feature_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
except TypeError:
    feature_encoder = OneHotEncoder(sparse=False, handle_unknown="ignore")

X_static_enc = feature_encoder.fit_transform(X_static)

label_encoder = LabelEncoder()
y_static_enc = label_encoder.fit_transform(y_static)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_static_enc, y_static_enc,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y_static_enc
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_temp
)

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

print("\nComputation graph:")
print("Input -> Linear -> ReLU -> Linear -> ReLU -> Output logits -> Softmax probabilities")
print("The model is trained with cross-entropy loss and gradient-based optimization.")

# ============================================================
# 7. PyTorch tabular dataset helpers
# ============================================================

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = TabularDataset(X_train, y_train)
val_ds = TabularDataset(X_val, y_val)
test_ds = TabularDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

# ============================================================
# 8. Feedforward neural network
# ============================================================

class FeedForwardNet(nn.Module):
    def __init__(self, input_dim: int, num_classes: int, hidden_dims=(64, 32), dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dims[1], num_classes)
        )

    def forward(self, x):
        return self.net(x)


def evaluate_model(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = model(xb)
            loss = loss_fn(logits, yb)

            total_loss += loss.item() * xb.size(0)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(yb.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average="macro")

    return avg_loss, acc, macro_f1


def train_feedforward_model(
    model,
    train_loader,
    val_loader,
    lr=0.01,
    weight_decay=0.0,
    max_epochs=40,
    patience=8
):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "train_f1": [],
        "val_f1": []
    }

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        total_train_loss = 0.0
        train_preds = []
        train_targets = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() * xb.size(0)
            preds = torch.argmax(logits, dim=1)

            train_preds.extend(preds.detach().cpu().numpy())
            train_targets.extend(yb.detach().cpu().numpy())

        train_loss = total_train_loss / len(train_loader.dataset)
        train_acc = accuracy_score(train_targets, train_preds)
        train_f1 = f1_score(train_targets, train_preds, average="macro")

        val_loss, val_acc, val_f1 = evaluate_model(model, val_loader, loss_fn)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_f1)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
            f"train_acc={train_acc:.4f} val_acc={val_acc:.4f} | "
            f"train_f1={train_f1:.4f} val_f1={val_f1:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print("Early stopping triggered.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history

# ============================================================
# 9. Train baseline feedforward model
# ============================================================

num_classes = len(label_encoder.classes_)
input_dim = X_train.shape[1]

ff_model = FeedForwardNet(
    input_dim=input_dim,
    num_classes=num_classes,
    hidden_dims=(64, 32),
    dropout=0.0
).to(DEVICE)

ff_model, ff_history = train_feedforward_model(
    ff_model,
    train_loader,
    val_loader,
    lr=0.01,
    weight_decay=0.0,
    max_epochs=40,
    patience=8
)

loss_fn = nn.CrossEntropyLoss()
test_loss, test_acc, test_f1 = evaluate_model(ff_model, test_loader, loss_fn)

print("\nBaseline Feedforward Test Performance")
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test macro-F1: {test_f1:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(ff_history["train_loss"], label="Train Loss")
plt.plot(ff_history["val_loss"], label="Validation Loss")
plt.title("Feedforward Network — Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("module7_ff_loss_curves.png", dpi=300, bbox_inches="tight")
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(ff_history["train_acc"], label="Train Accuracy")
plt.plot(ff_history["val_acc"], label="Validation Accuracy")
plt.title("Feedforward Network — Accuracy Curves")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("module7_ff_accuracy_curves.png", dpi=300, bbox_inches="tight")
plt.close()

print("\nSaved:")
print("  module7_ff_loss_curves.png")
print("  module7_ff_accuracy_curves.png")

# ============================================================
# 10. Part 2 — Optimization experiments
# ============================================================

print("\n" + "=" * 70)
print("PART 2 — OPTIMIZATION (21.4)")
print("=" * 70)

learning_rates = [0.1, 0.01, 0.001]
lr_results = {}

for lr in learning_rates:
    print(f"\n=== Training with learning rate = {lr} ===")

    model_lr = FeedForwardNet(
        input_dim=input_dim,
        num_classes=num_classes,
        hidden_dims=(64, 32),
        dropout=0.0
    ).to(DEVICE)

    model_lr, hist_lr = train_feedforward_model(
        model_lr,
        train_loader,
        val_loader,
        lr=lr,
        weight_decay=0.0,
        max_epochs=25,
        patience=6
    )

    val_loss, val_acc, val_f1 = evaluate_model(model_lr, val_loader, nn.CrossEntropyLoss())
    lr_results[lr] = {
        "history": hist_lr,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1
    }

print("\nLearning rate results summary:")
for lr, result in lr_results.items():
    print(
        f"lr={lr:<6} "
        f"val_loss={result['val_loss']:.4f} "
        f"val_acc={result['val_acc']:.4f} "
        f"val_f1={result['val_f1']:.4f}"
    )

plt.figure(figsize=(8, 5))
for lr in learning_rates:
    plt.plot(lr_results[lr]["history"]["val_loss"], label=f"lr={lr}")
plt.title("Optimization Experiment — Validation Loss by Learning Rate")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("module7_learning_rate_comparison.png", dpi=300, bbox_inches="tight")
plt.close()

print("\nOptimization analysis:")
print("- High learning rates may overshoot and become unstable.")
print("- Low learning rates may converge slowly.")
print("- Moderate learning rates often provide the best balance.")

print("\nSaved:")
print("  module7_learning_rate_comparison.png")

# ============================================================
# 11. Part 3 — Generalization: overfitting and regularization
# ============================================================

print("\n" + "=" * 70)
print("PART 3 — GENERALIZATION (21.5)")
print("=" * 70)

X_small, _, y_small, _ = train_test_split(
    X_train,
    y_train,
    train_size=3000,
    random_state=RANDOM_SEED,
    stratify=y_train
)

small_train_ds = TabularDataset(X_small, y_small)
small_train_loader = DataLoader(small_train_ds, batch_size=128, shuffle=True)

print("Small training subset for overfitting demo:", X_small.shape)

# Overfitting model
overfit_model = FeedForwardNet(
    input_dim=input_dim,
    num_classes=num_classes,
    hidden_dims=(256, 128),
    dropout=0.0
).to(DEVICE)

overfit_model, overfit_history = train_feedforward_model(
    overfit_model,
    small_train_loader,
    val_loader,
    lr=0.01,
    weight_decay=0.0,
    max_epochs=60,
    patience=12
)

# Regularized model
regularized_model = FeedForwardNet(
    input_dim=input_dim,
    num_classes=num_classes,
    hidden_dims=(64, 32),
    dropout=0.30
).to(DEVICE)

regularized_model, regularized_history = train_feedforward_model(
    regularized_model,
    small_train_loader,
    val_loader,
    lr=0.01,
    weight_decay=1e-4,
    max_epochs=60,
    patience=12
)

plt.figure(figsize=(8, 5))
plt.plot(overfit_history["train_loss"], label="Overfit Model - Train Loss")
plt.plot(overfit_history["val_loss"], label="Overfit Model - Val Loss")
plt.title("Overfitting Demonstration")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("module7_overfitting_demo.png", dpi=300, bbox_inches="tight")
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(regularized_history["train_loss"], label="Regularized Model - Train Loss")
plt.plot(regularized_history["val_loss"], label="Regularized Model - Val Loss")
plt.title("Regularization Demonstration")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("module7_regularization_demo.png", dpi=300, bbox_inches="tight")
plt.close()

overfit_test = evaluate_model(overfit_model, test_loader, nn.CrossEntropyLoss())
regularized_test = evaluate_model(regularized_model, test_loader, nn.CrossEntropyLoss())

print("\nOverfit model test performance:")
print(f"loss={overfit_test[0]:.4f}, acc={overfit_test[1]:.4f}, macro_f1={overfit_test[2]:.4f}")

print("\nRegularized model test performance:")
print(f"loss={regularized_test[0]:.4f}, acc={regularized_test[1]:.4f}, macro_f1={regularized_test[2]:.4f}")

print("\nGeneralization analysis:")
print("- Overfitting appears when training performance improves faster than validation performance.")
print("- Dropout, weight decay, and early stopping improve generalization.")

print("\nSaved:")
print("  module7_overfitting_demo.png")
print("  module7_regularization_demo.png")

# ============================================================
# 12. Part 4 — Advanced architecture: RNN
# ============================================================

print("\n" + "=" * 70)
print("PART 4 — ADVANCED ARCHITECTURE (RNN)")
print("=" * 70)

all_locations_for_rnn = ["START"] + MAIN_LOCATIONS
loc_to_idx = {loc: i for i, loc in enumerate(all_locations_for_rnn)}
idx_to_loc = {i: loc for loc, i in loc_to_idx.items()}

type_to_idx = {t: i for i, t in enumerate(STUDENT_TYPES)}
sport_to_idx = {s: i for i, s in enumerate(SPORTS)}
day_to_idx = {d: i for i, d in enumerate(DAY_OF_WEEK)}

sequence_rows = []

for (sid, day), group in df_seq.groupby(["student_id", "day_of_week"]):
    group = group.set_index("time_slot").loc[TIME_SLOTS].reset_index()

    student_type = group["student_type"].iloc[0]
    sport = group["sport"].iloc[0]
    main_seq = group["main_location"].tolist()

    prefix = ["START"]
    for next_loc in main_seq:
        sequence_rows.append({
            "student_type_idx": type_to_idx[student_type],
            "sport_idx": sport_to_idx[sport],
            "day_idx": day_to_idx[day],
            "prefix_seq": [loc_to_idx[x] for x in prefix],
            "target_loc": loc_to_idx[next_loc]
        })
        prefix.append(next_loc)

print("Number of RNN samples:", len(sequence_rows))
print("Example sequence sample:", sequence_rows[0])

class SequenceDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        return (
            torch.tensor(row["student_type_idx"], dtype=torch.long),
            torch.tensor(row["sport_idx"], dtype=torch.long),
            torch.tensor(row["day_idx"], dtype=torch.long),
            torch.tensor(row["prefix_seq"], dtype=torch.long),
            torch.tensor(row["target_loc"], dtype=torch.long)
        )


def collate_sequence_batch(batch):
    type_idxs, sport_idxs, day_idxs, seqs, targets = zip(*batch)

    lengths = [len(s) for s in seqs]
    max_len = max(lengths)

    padded = torch.zeros(len(seqs), max_len, dtype=torch.long)
    for i, seq in enumerate(seqs):
        padded[i, :len(seq)] = seq

    return (
        torch.stack(type_idxs),
        torch.stack(sport_idxs),
        torch.stack(day_idxs),
        padded,
        torch.tensor(lengths, dtype=torch.long),
        torch.stack(targets)
    )

indices = np.arange(len(sequence_rows))
train_idx, temp_idx = train_test_split(indices, test_size=0.30, random_state=RANDOM_SEED)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=RANDOM_SEED)

rows_train = [sequence_rows[i] for i in train_idx]
rows_val = [sequence_rows[i] for i in val_idx]
rows_test = [sequence_rows[i] for i in test_idx]

rnn_train_ds = SequenceDataset(rows_train)
rnn_val_ds = SequenceDataset(rows_val)
rnn_test_ds = SequenceDataset(rows_test)

rnn_train_loader = DataLoader(rnn_train_ds, batch_size=128, shuffle=True, collate_fn=collate_sequence_batch)
rnn_val_loader = DataLoader(rnn_val_ds, batch_size=128, shuffle=False, collate_fn=collate_sequence_batch)
rnn_test_loader = DataLoader(rnn_test_ds, batch_size=128, shuffle=False, collate_fn=collate_sequence_batch)

class CampusRNN(nn.Module):
    def __init__(
        self,
        num_locations,
        num_types,
        num_sports,
        num_days,
        loc_emb_dim=16,
        meta_emb_dim=4,
        hidden_dim=64
    ):
        super().__init__()

        self.loc_embedding = nn.Embedding(num_locations, loc_emb_dim)
        self.type_embedding = nn.Embedding(num_types, meta_emb_dim)
        self.sport_embedding = nn.Embedding(num_sports, meta_emb_dim)
        self.day_embedding = nn.Embedding(num_days, meta_emb_dim)

        self.rnn = nn.GRU(
            input_size=loc_emb_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim + 3 * meta_emb_dim, 64),
            nn.ReLU(),
            nn.Linear(64, len(all_locations_for_rnn))
        )

    def forward(self, type_idx, sport_idx, day_idx, seqs, lengths):
        loc_emb = self.loc_embedding(seqs)
        _, hidden = self.rnn(loc_emb)
        last_hidden = hidden[-1]

        type_emb = self.type_embedding(type_idx)
        sport_emb = self.sport_embedding(sport_idx)
        day_emb = self.day_embedding(day_idx)

        x = torch.cat([last_hidden, type_emb, sport_emb, day_emb], dim=1)
        logits = self.fc(x)
        return logits


def evaluate_rnn(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    preds_all = []
    targets_all = []

    with torch.no_grad():
        for type_idx, sport_idx, day_idx, seqs, lengths, targets in loader:
            type_idx = type_idx.to(DEVICE)
            sport_idx = sport_idx.to(DEVICE)
            day_idx = day_idx.to(DEVICE)
            seqs = seqs.to(DEVICE)
            lengths = lengths.to(DEVICE)
            targets = targets.to(DEVICE)

            logits = model(type_idx, sport_idx, day_idx, seqs, lengths)
            loss = loss_fn(logits, targets)

            total_loss += loss.item() * targets.size(0)
            preds = torch.argmax(logits, dim=1)

            preds_all.extend(preds.cpu().numpy())
            targets_all.extend(targets.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(targets_all, preds_all)
    macro_f1 = f1_score(targets_all, preds_all, average="macro")

    return avg_loss, acc, macro_f1


def train_rnn_model(model, train_loader, val_loader, lr=0.005, max_epochs=30, patience=6):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0.0
        preds_all = []
        targets_all = []

        for type_idx, sport_idx, day_idx, seqs, lengths, targets in train_loader:
            type_idx = type_idx.to(DEVICE)
            sport_idx = sport_idx.to(DEVICE)
            day_idx = day_idx.to(DEVICE)
            seqs = seqs.to(DEVICE)
            lengths = lengths.to(DEVICE)
            targets = targets.to(DEVICE)

            optimizer.zero_grad()
            logits = model(type_idx, sport_idx, day_idx, seqs, lengths)
            loss = loss_fn(logits, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * targets.size(0)
            preds = torch.argmax(logits, dim=1)

            preds_all.extend(preds.detach().cpu().numpy())
            targets_all.extend(targets.detach().cpu().numpy())

        train_loss = total_loss / len(train_loader.dataset)
        train_acc = accuracy_score(targets_all, preds_all)
        val_loss, val_acc, _ = evaluate_rnn(model, val_loader, loss_fn)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
            f"train_acc={train_acc:.4f} val_acc={val_acc:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print("Early stopping triggered.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history

rnn_model = CampusRNN(
    num_locations=len(all_locations_for_rnn),
    num_types=len(STUDENT_TYPES),
    num_sports=len(SPORTS),
    num_days=len(DAY_OF_WEEK),
    loc_emb_dim=16,
    meta_emb_dim=4,
    hidden_dim=64
).to(DEVICE)

rnn_model, rnn_history = train_rnn_model(
    rnn_model,
    rnn_train_loader,
    rnn_val_loader,
    lr=0.005,
    max_epochs=30,
    patience=6
)

rnn_test_loss, rnn_test_acc, rnn_test_f1 = evaluate_rnn(
    rnn_model,
    rnn_test_loader,
    nn.CrossEntropyLoss()
)

print("\nRNN Test Performance")
print(f"Test loss    : {rnn_test_loss:.4f}")
print(f"Test accuracy: {rnn_test_acc:.4f}")
print(f"Test macro-F1: {rnn_test_f1:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(rnn_history["train_loss"], label="Train Loss")
plt.plot(rnn_history["val_loss"], label="Validation Loss")
plt.title("RNN — Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("module7_rnn_loss_curves.png", dpi=300, bbox_inches="tight")
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(rnn_history["train_acc"], label="Train Accuracy")
plt.plot(rnn_history["val_acc"], label="Validation Accuracy")
plt.title("RNN — Accuracy Curves")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("module7_rnn_accuracy_curves.png", dpi=300, bbox_inches="tight")
plt.close()

print("\nAdvanced architecture analysis:")
print("- The feedforward model uses only static inputs.")
print("- The RNN uses temporal context from previous locations.")
print("- This makes the RNN a more natural fit for campus movement sequences.")

print("\nSaved:")
print("  module7_rnn_loss_curves.png")
print("  module7_rnn_accuracy_curves.png")

# ============================================================
# 13. Compare baseline vs RNN
# ============================================================

print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

comparison_df = pd.DataFrame({
    "Model": ["Feedforward Baseline", "RNN Sequential Model"],
    "Test Loss": [test_loss, rnn_test_loss],
    "Test Accuracy": [test_acc, rnn_test_acc],
    "Test Macro-F1": [test_f1, rnn_test_f1]
})

print(comparison_df)

# ============================================================
# 14. Part 5 — Application design
# ============================================================

print("\n" + "=" * 70)
print("PART 5 — APPLICATION DESIGN (21.7–21.8)")
print("=" * 70)

def predict_static_location_distribution(student_type, sport, day_of_week, time_slot):
    """
    User-facing application helper:
    Given dropdown selections, return probabilities over main locations.
    """
    x_df = pd.DataFrame([{
        "student_type": student_type,
        "sport": sport,
        "day_of_week": day_of_week,
        "time_slot": time_slot
    }])

    x_enc = feature_encoder.transform(x_df)
    x_tensor = torch.tensor(x_enc, dtype=torch.float32).to(DEVICE)

    ff_model.eval()
    with torch.no_grad():
        logits = ff_model(x_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    output = pd.DataFrame({
        "main_location": label_encoder.classes_,
        "probability": probs
    }).sort_values("probability", ascending=False)

    return output


example_prediction = predict_static_location_distribution(
    student_type="Athlete",
    sport="TrackAndField",
    day_of_week="Tuesday",
    time_slot="16:00-17:30"
)

print("Example application output:")
print(example_prediction.head(10))

print("\nApplication design summary:")
print("- Static mode: predict P(main_location | type, sport, day, time)")
print("- Sequential mode: RNN predicts next likely location from prior locations")
print("- This can support facility planning, staffing, and campus movement analysis")

# ============================================================
# 15. Final classification report for baseline model
# ============================================================

print("\n" + "=" * 70)
print("FINAL PERFORMANCE METRICS")
print("=" * 70)

ff_model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        logits = ff_model(xb)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(yb.numpy())

print("Feedforward classification report:")
print(classification_report(
    all_targets,
    all_preds,
    target_names=label_encoder.classes_,
    zero_division=0
))

# ============================================================
# 16. Markdown-ready explanation block
# ============================================================

print("\n\n--- Markdown-ready explanation (paste into notebook or report) ---\n")
print(r"""
### Module 7 — Deep Learning for Campus Movement Prediction

This module extends the same campus movement world used in Module 6. Earlier work focused on probabilistic learning, including maximum likelihood estimation, conditional probability learning, and EM for hidden activity modes. In this module, we build deep learning models for prediction and sequential modeling.

### Part 1 — Feedforward model
A feedforward neural network is trained to predict the main campus location from:
- student type
- sport
- day of week
- time slot

The computation graph is:

\[
x \rightarrow \text{Linear} \rightarrow \text{ReLU} \rightarrow \text{Linear} \rightarrow \text{ReLU} \rightarrow \text{Output logits} \rightarrow \text{Softmax}
\]

The model is trained with cross-entropy loss and gradient-based optimization.

### Part 2 — Optimization
To study optimization behavior, the same feedforward architecture is trained with multiple learning rates. This demonstrates how training convergence depends on the step size used in gradient-based learning:
- very large learning rates can be unstable
- very small learning rates can converge slowly
- moderate learning rates often provide the best balance

### Part 3 — Generalization
Generalization is analyzed by deliberately creating an overfitting scenario using:
- a larger neural network
- a smaller training subset

Regularization is then applied using:
- dropout
- weight decay
- early stopping

This shows how regularization helps reduce overfitting and improve validation/test behavior.

### Part 4 — Advanced architecture
A recurrent neural network (GRU-based) is trained on student-day movement sequences. Instead of using only static features, the RNN predicts the next location from:
- previous locations earlier in the day
- student type
- sport
- day of week

This is a natural extension of the project because campus movement is sequential.

### Part 5 — Application design
A simple application interface can use the trained feedforward model to return:

\[
P(\text{main location} \mid \text{student type}, \text{sport}, \text{day}, \text{time})
\]

For example, a user can select:
- Athlete
- TrackAndField
- Tuesday
- 16:00-17:30

and the model returns a ranked probability distribution over campus locations.

### Integrated sports schedules
This module includes explicit athlete schedule structure for:
- Track and Field
- Baseball

This makes the dataset more realistic and allows the model to distinguish between different athlete movement patterns rather than treating all athletes as homogeneous.

### Summary
Module 7 satisfies the deep learning requirements by including:
- a feedforward neural network
- gradient-based training
- optimization experiments
- generalization analysis
- an advanced recurrent architecture
- a real-world application design component

It also remains fully consistent with the semester-long campus movement project.
""".strip())

print("\nAll done.")

PyTorch version: 2.10.0
CUDA available: False
Using device: cpu

DATA GENERATION
Static dataset shape: (40000, 6)
      student_type          sport day_of_week    time_slot     main_location  \
0             Copa           None     Tuesday  16:00-17:30  BoulevardStudios   
1          Athlete  TrackAndField      Monday  15:00-15:30           Library   
2  Regular Student           None    Thursday  16:00-17:30     StudentCenter   
3             Copa           None      Monday  08:00-09:30         OffCampus   
4          Athlete  TrackAndField      Monday  16:00-17:30       VillagePark   

     inner_location  
0  BoulevardStudios  
1        StudyRooms  
2          GameRoom  
3         OffCampus  
4       VillagePark  

Sequential dataset shape: (100000, 8)
   student_id student_type sport day_of_week    time_slot prev_main_location  \
0           0         Copa  None      Monday  07:00-08:00              START   
1           0         Copa  None      Monday  08:00-09:30          OffCamp